# 01 - Let's design an A/B test

**Business Loves Uplift: Causal Impact Evaluation With Python**<br>
ODSC AI West 2026 · Alex Molak · [CausalPython.io](https://causalpython.io)

<a href="https://colab.research.google.com/github/AlxndrMlk/talks-and-workshops/blob/main/2026-10--odsc-ai/01%20-%20AB%20test.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>


**The story.** A retailer's field sales reps each look after 120-200 stores. The data team built a recommender that tells a rep which new products (SKUs) to list in each store. How much does it lift store sales?

The data is synthetic, so we know the true answer. Weekly sales are in k$ per store.

In this notebook we design the simplest test: flip a coin for every store. We run a power analysis, analyze the test, and then check the answer against the truth.

In [ ]:
# Colab only: install the packages and get the data generator (locally, use the workshop env)
import sys
if "google.colab" in sys.modules:
    !pip install -q dowhy==0.14 econml==0.17.0 cluster-experiments==0.31.0 mlsynth==1.0.0
    !wget -q -nc https://raw.githubusercontent.com/AlxndrMlk/talks-and-workshops/main/2026-10--odsc-ai/uplift_data.py

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from cluster_experiments import NormalPowerAnalysis, OLSAnalysis

import uplift_data as ud

BLUE, GOLD, GREY, RED, DARK = "#0057d9", "#c8860d", "#bbbbbb", "#c0392b", "#222222"
plt.rcParams.update({"figure.figsize": (9, 4), "axes.spines.top": False, "axes.spines.right": False})

In [ ]:
d = ud.make_datasets()     # seed 23: everyone gets the same world
truth = d["truth"]         # the answer key; we only peek at it at the end of each section

## The data before the test

`history` holds weekly sales of every store for the 8 weeks before the test. Nobody has the recommender yet.

In [ ]:
history = d["history"]
print(f"{history.store_id.nunique()} stores, {history.rep_id.nunique()} reps, {history.week.nunique()} weeks")
history.head()

In [ ]:
# one row per store: mean weekly sales over the 8 weeks
stores = history.groupby(["store_id", "rep_id"], as_index=False).sales.mean()

plt.hist(stores.sales, bins=60, color=GREY)
plt.xlabel("mean weekly sales per store (k$)")
plt.ylabel("stores")
plt.show()

## Design: randomize stores

The budget allows a test with 40 reps. In this design, every store of these reps flips a coin: heads gets the recommender, tails doesn't.

How small an effect can such a test detect? `cluster-experiments` answers this by simulation on the historical data: it splits the stores at random many times, analyzes each split, and reads off the minimum detectable effect (MDE) at 80% power.

In [ ]:
d['ab_store']

In [ ]:
test_reps = d["ab_store"].rep_id.unique()     # the 40 reps in the test
stores_test = stores[stores.rep_id.isin(test_reps)]
print(f"{len(test_reps)} reps, {len(stores_test)} stores, mean sales {stores_test.sales.mean():.1f} k$ per week")

In [ ]:
pw_store = NormalPowerAnalysis.from_dict({
    "splitter": "non_clustered",   # each store flips its own coin
    "analysis": "ols",             # difference in means, regular standard errors
    "cluster_cols": ["store_id"],
    "target_col": "sales",
    "n_simulations": 50,
    "seed": 0,
})

mde = pw_store.mde(stores_test, power=0.8)
print(f"MDE at 80% power: {mde:.2f} k$ = {mde / stores_test.sales.mean():.1%} of sales")

In [ ]:
effects = np.linspace(0, 0.6, 13)
power = pw_store.power_line(stores_test, average_effects=list(effects))

plt.plot(list(power.keys()), list(power.values()), "o-", color=BLUE)
plt.axhline(0.8, ls="--", color=GREY)
plt.xlabel("true effect (k$ per store-week)")
plt.ylabel("power")
plt.title("Store-level design: power vs effect size")
plt.show()

Thousands of stores, an MDE under 2% of sales: this looks like an easy test. Let's run it.

## Run and analyze the test

`ab_store` has one row per test store, with mean weekly sales before the test (`sales_pre`, weeks 0-7) and during it (`sales_test`, weeks 8-15).

In [ ]:
ab = d["ab_store"]

ab

In [ ]:
means = ab.groupby("treated").sales_test.mean()
print(f"control {means[0]:.2f} k$, treatment {means[1]:.2f} k$, difference {means[1] - means[0]:.2f} k$")

We analyze the test with the same package and the same analysis (`OLSAnalysis`) we planned the power for.

In [ ]:
analysis = OLSAnalysis(target_col="sales_test", treatment_col="treated", treatment=1)

In [ ]:
r = analysis.get_inference_results(ab, alpha=0.05)
est, lo, hi = r.ate, r.conf_int.lower, r.conf_int.upper
base = ab.loc[ab.treated == 0, "sales_test"].mean()
print(f"effect: {est:.2f} k$ [{lo:.2f}, {hi:.2f}]  =  +{est / base:.1%} of control sales")

The same analysis in plain statsmodels (`OLSAnalysis` uses robust HC3 standard errors by default):

In [ ]:
fit = smf.ols("sales_test ~ treated", ab).fit(cov_type="HC3")
print(f"effect: {fit.params['treated']:.2f} k$ {fit.conf_int().loc['treated'].round(2).tolist()}")

## Answer key

We built the data, so we know the true effect of the recommender on the stores that got it.

In [ ]:
direct = truth["ab_store_direct_abs"]        # true effect on treated stores
leak = truth["ab_store_control_spill_abs"]   # spillover: what untreated control stores gained via their rep
print(f"true effect on treated stores:    {direct:.2f} k$")
print(f"our estimate:                     {est:.2f} k$  [{lo:.2f}, {hi:.2f}]")
print(f"spillover onto control stores:    {leak:.2f} k$")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.barh(["effect on treated stores", "effect leaked onto control stores", "what the test measures"],
        [direct, leak, direct - leak], color=[BLUE, GREY, DARK])
ax.errorbar(est, 2, xerr=[[est - lo], [hi - est]], fmt="o", color=RED, capsize=4, label="our estimate (95% CI)")
ax.invert_yaxis()
ax.set_xlabel("k$ per store-week")
ax.legend(loc="upper right", bbox_to_anchor=(1, -0.25), frameon=False)
plt.show()

The test misses the true effect, and its confidence interval doesn't even cover it.

**Why?** The treatment isn't a store's: it's the rep's. A rep who sees the recommendations on half of their stores starts listing the same products in the other half too. The control stores get part of the effect (*interference*), so the difference between the arms shrinks.

There's a second problem, which we'll see in the next notebook: stores of the same rep move together, so the regular standard errors are too small, and the MDE above is too optimistic.

The fix for both: randomize **reps**, not stores.